> ## SOLUTION / ANSWER KEY &mdash; Lab 6.7
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-07-conversation-memory.ipynb`](../lab-07-conversation-memory.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.7 &mdash; Memory: Conversation History

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Store each turn (role, text) in a memory object
- Format the running history the model re-reads each turn
- Build a prompt that carries context so a follow-up resolves

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps use a tiny **LangChain-shaped shim** (same names &amp; shapes as real LangChain) so they run offline &amp; deterministically. Advanced labs end with an **optional** cell that runs the **real** library.

**Reference:** [Module 6 slides &mdash; Memory -- plugged in, not hand-built](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-06-07"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
A framework keeps **conversation memory** so follow-ups work: ask *"capital of France?"*, then
*"and Osaka?"* &mdash; the agent needs the earlier turns to resolve *"and"*. Memory is a component you
**attach**: it stores turns and re-injects the **history** into each prompt. (Long-term memory is a
vector store &mdash; the bridge to RAG.)

In [ ]:
class AIMessage:
    def __init__(self, content): self.content = content
class FakeChatModel:
    """Deterministic stand-in for ChatOllama / ChatGroq: replays a scripted list of replies.
    Real code: from langchain_ollama import ChatOllama; model = ChatOllama(model="llama3.2:1b").
    Like the real thing, .invoke(prompt) returns a message whose text is in .content."""
    def __init__(self, script): self.script = list(script); self.i = 0
    def invoke(self, prompt):
        reply = self.script[min(self.i, len(self.script) - 1)]; self.i += 1
        return AIMessage(reply)

class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

# one turn is just (role, text)
print("a turn:", ("user", "What is the capital of France?"))

## Your Turn
Implement `ConversationMemory` (store + format the history) and a prompt that injects it.

In [ ]:
class ConversationMemory:
    def __init__(self):
        self.turns = []
    def add(self, role, text):
        self.turns.append((role, text))
    def history(self):
        return "\n".join(r + ": " + t for r, t in self.turns)

def build_prompt(memory, question):
    template = PromptTemplate.from_template("Conversation so far:\n{history}\nUser: {input}")
    return template.format(history=memory.history(), input=question)

try:
    mem = ConversationMemory()
    mem.add('user', 'What is the capital of France?')
    mem.add('ai', 'Paris.')
    followup = build_prompt(mem, 'And Osaka is in which country?')
    print(followup)
    print('carries prior context?', 'Paris' in followup)
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("memory stores each turn", lambda: (lambda m: (m.add("user", "hi"), len(m.turns))[1])(ConversationMemory()) == 1)
expect_true("history has one line per turn", lambda: (lambda m: (m.add("user", "a"), m.add("ai", "b"), m.history().count(chr(10)))[2])(ConversationMemory()) == 1)
expect_true("history formats role: text", lambda: (lambda m: (m.add("user", "hi"), m.history())[1])(ConversationMemory()) == "user: hi")
expect_true("the follow-up prompt carries prior context", lambda: "Paris" in (lambda m: (m.add("ai", "Paris."), build_prompt(m, "and Osaka?"))[1])(ConversationMemory()))
expect_true("the prompt includes the new question", lambda: "Osaka" in build_prompt(ConversationMemory(), "where is Osaka?"))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Memory carries context across turns so 'and Osaka?' just works. In a framework it's a component you attach -- not a scratchpad you hand-build.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>